Buffer the hedges

In [25]:
import arcpy
import os

arcpy.env.overwriteOutput = True

GDB = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb"
IN_LINES = os.path.join(GDB, "hedges_Clip")
OUT_BUF  = os.path.join(GDB, "hedges_Clip_buf_1p25m")

if arcpy.Exists(OUT_BUF):
    arcpy.management.Delete(OUT_BUF)

dist = "1.25 Meters"

# PairwiseBuffer is usually faster; fall back to Buffer if not available
if hasattr(arcpy.analysis, "PairwiseBuffer"):
    arcpy.analysis.PairwiseBuffer(
        in_features=IN_LINES,
        out_feature_class=OUT_BUF,
        buffer_distance_or_field=dist,
        dissolve_option="NONE"
    )
else:
    arcpy.analysis.Buffer(
        in_features=IN_LINES,
        out_feature_class=OUT_BUF,
        buffer_distance_or_field=dist,
        line_side="FULL",
        line_end_type="ROUND",
        dissolve_option="NONE",
        method="PLANAR"
    )

print(f"✅ Wrote: {OUT_BUF}")


✅ Wrote: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb\hedges_Clip_buf_1p25m


Now, combine it with merged crowns, removing overlaps, assigning LC = 23

In [26]:
import arcpy
import os

arcpy.env.overwriteOutput = True

GDB = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb"

CROWNS = os.path.join(GDB, "Crowns_RGB_LiDAR_merged")
HEDGES = os.path.join(GDB, "hedges_Clip_buf_1p25m")

HEDGES_OUTSIDE = os.path.join(GDB, "hedges_buf_outside_crowns")
OUT_COMBINED   = os.path.join(GDB, "Crowns_plus_hedges_nooverlap")

# cleanup
for fc in [HEDGES_OUTSIDE, OUT_COMBINED]:
    if arcpy.Exists(fc):
        arcpy.management.Delete(fc)

# 1) Erase hedges by crowns (keep only hedge parts OUTSIDE crowns)
if hasattr(arcpy.analysis, "PairwiseErase"):
    arcpy.analysis.PairwiseErase(HEDGES, CROWNS, HEDGES_OUTSIDE)
else:
    arcpy.analysis.Erase(HEDGES, CROWNS, HEDGES_OUTSIDE)

# 2) Ensure LC field and set LC = 23 on hedges_outside
if "LC" not in {f.name.upper() for f in arcpy.ListFields(HEDGES_OUTSIDE)}:
    arcpy.management.AddField(HEDGES_OUTSIDE, "LC", "LONG")
arcpy.management.CalculateField(HEDGES_OUTSIDE, "LC", "23", "PYTHON3")

# Optional: tag source
if "SOURCE" not in {f.name.upper() for f in arcpy.ListFields(HEDGES_OUTSIDE)}:
    arcpy.management.AddField(HEDGES_OUTSIDE, "SOURCE", "TEXT", field_length=10)
arcpy.management.CalculateField(HEDGES_OUTSIDE, "SOURCE", '"HEDGE"', "PYTHON3")

# 3) Merge crowns + hedges_outside (now no overlaps)
arcpy.management.Merge([CROWNS, HEDGES_OUTSIDE], OUT_COMBINED)

print(f"✅ Hedges outside crowns: {HEDGES_OUTSIDE}")
print(f"✅ Combined no-overlap:   {OUT_COMBINED}")


✅ Hedges outside crowns: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb\hedges_buf_outside_crowns
✅ Combined no-overlap:   C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb\Crowns_plus_hedges_nooverlap


Now I need to get a 

In [11]:
# =============================================================================
# HEDGEROW AGB (LC=23) FROM CHM: polygons + random sampling (Monte Carlo)
# - Uses IndexerKey as the stable join key (recommended)
# - Writes into AGB_KG on fc_all, only for LC=23 (hedges)
# =============================================================================

import os
import arcpy
from arcpy.sa import ExtractValuesToPoints

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

# ----------------------------
# INPUTS
# ----------------------------
gdb = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb"
fc_all = os.path.join(gdb, "Crowns_plus_hedges_nooverlap")
chm    = os.path.join(gdb, "chm_raw_CopyRaste")

# ----------------------------
# PARAMETERS
# ----------------------------
LC_FIELD     = "LC"
HEDGE_LC     = 23
KEY_FIELD    = "IndexerKey"   # <- your stable ID
AGB_FIELD    = "AGB_KG"

ONLY_IF_NULL = False           # don't recalc if AGB already populated
TREAT_ZERO_AS_MISSING = True # optional

SPACING_M = 1.0               # SRUC-ish min distance (set 1.0 if you really want dense)
H_MIN_M   = 1               # SRUC-ish cutoff (heights <= 1.3m contribute 0)

SCALE_TO_AREA = True
N_ITERS = 20
SAFETY  = 0.25
MAX_POINTS = 200000           # cap to prevent huge point counts; set None to disable

# point model: AGB = a * h^b
AGB_A = 0.179
AGB_B = 3.3

KEEP_INTERMEDIATES = False

# ----------------------------
# SCRATCH OUTPUTS
# ----------------------------
scratch = arcpy.env.scratchGDB or gdb
hedge_poly = os.path.join(scratch, "hedge_lc23_sel")
pts_rand   = os.path.join(scratch, "hedge_pts_rand")
pts_samp   = os.path.join(scratch, "hedge_pts_samp")
pt_poly    = os.path.join(scratch, "hedge_pts_joinpoly")
sum_tbl    = os.path.join(scratch, "hedge_agb_sumtbl")

def safe_delete(p):
    if arcpy.Exists(p):
        arcpy.management.Delete(p)

# ----------------------------
# CHECKS
# ----------------------------
if not arcpy.Exists(fc_all):
    raise RuntimeError(f"Missing feature class: {fc_all}")
if not arcpy.Exists(chm):
    raise RuntimeError(f"Missing raster: {chm}")

fields_upper = {f.name.upper(): f for f in arcpy.ListFields(fc_all)}

for req in [LC_FIELD, KEY_FIELD]:
    if req.upper() not in fields_upper:
        raise RuntimeError(f"Required field missing on fc_all: {req}")

# Ensure AGB field exists
if AGB_FIELD.upper() not in fields_upper:
    arcpy.management.AddField(fc_all, AGB_FIELD, "DOUBLE")
    fields_upper = {f.name.upper(): f for f in arcpy.ListFields(fc_all)}

# LC field quoting (text vs numeric)
lc_type = fields_upper[LC_FIELD.upper()].type.lower()
lc_is_text = lc_type in ("string", "text")
lc_val_sql = f"'{HEDGE_LC}'" if lc_is_text else str(HEDGE_LC)

# Where clause: LC=23 (and optional AGB NULL)
where = f"{arcpy.AddFieldDelimiters(fc_all, LC_FIELD)} = {lc_val_sql}"
if ONLY_IF_NULL:
    agb_delim = arcpy.AddFieldDelimiters(fc_all, AGB_FIELD)
    if TREAT_ZERO_AS_MISSING:
        where += f" AND ({agb_delim} IS NULL OR {agb_delim} = 0)"
    else:
        where += f" AND {agb_delim} IS NULL"

print("Where clause:", where)

# ENV: align to CHM; output SR = FC SR (safer)
arcpy.env.snapRaster = chm
arcpy.env.cellSize   = chm
arcpy.env.outputCoordinateSystem = arcpy.Describe(fc_all).spatialReference

# ----------------------------
# 1) Select hedge polys (LC=23) into scratch copy (keeping IndexerKey)
# ----------------------------
for p in [hedge_poly, pts_rand, pts_samp, pt_poly, sum_tbl]:
    safe_delete(p)

arcpy.analysis.Select(fc_all, hedge_poly, where)

n_hedges = int(arcpy.management.GetCount(hedge_poly)[0])
print(f"Hedge polygons selected: {n_hedges:,}")
if n_hedges == 0:
    raise SystemExit("Nothing to do (no hedges matched selection).")

# ----------------------------
# 2) Precompute area by IndexerKey (for scaling)
# ----------------------------
area_by_key = {}
total_area = 0.0

with arcpy.da.SearchCursor(hedge_poly, [KEY_FIELD, "SHAPE@AREA"]) as cur:
    for k, a in cur:
        if k is None or a is None:
            continue
        a = float(a)
        if a <= 0:
            continue
        k = int(k)
        area_by_key[k] = area_by_key.get(k, 0.0) + a
        total_area += a

expN_by_key = {k: (a / (SPACING_M * SPACING_M)) for k, a in area_by_key.items()}

n_points = max(1, int(total_area / (SPACING_M * SPACING_M * SAFETY)))
if MAX_POINTS is not None:
    n_points = min(n_points, int(MAX_POINTS))

print(f"Hedge area (m²): {total_area:,.1f}")
print(f"Requested points per iter: {n_points:,}  (spacing={SPACING_M} m, safety={SAFETY})")

# ----------------------------
# Helper: CHM height -> point AGB
# ----------------------------
def calc_point_agb(in_pts):
    fset = {f.name.upper() for f in arcpy.ListFields(in_pts)}
    if "AGB_KG_PT" not in fset:
        arcpy.management.AddField(in_pts, "AGB_KG_PT", "DOUBLE")

    codeblock = f"""
def agb(h):
    if h is None:
        return None
    if h <= {H_MIN_M}:
        return 0.0
    return {AGB_A} * (h ** {AGB_B})
"""
    arcpy.management.CalculateField(
        in_table=in_pts,
        field="AGB_KG_PT",
        expression="agb(!RASTERVALU!)",
        expression_type="PYTHON3",
        code_block=codeblock
    )

# ----------------------------
# 3) Monte Carlo sampling
# ----------------------------
sum_agb_by_key = {k: 0.0 for k in area_by_key.keys()}
valid_iters = 0

iters = range(int(N_ITERS))
if tqdm:
    iters = tqdm(iters, desc="Hedge AGB iters")

for _ in iters:
    for p in [pts_rand, pts_samp, pt_poly, sum_tbl]:
        safe_delete(p)

    arcpy.management.CreateRandomPoints(
        out_path=scratch,
        out_name=os.path.basename(pts_rand),
        constraining_feature_class=hedge_poly,
        number_of_points_or_field=n_points,
        minimum_allowed_distance=f"{SPACING_M} Meters"
    )

    ExtractValuesToPoints(pts_rand, chm, pts_samp, "NONE")
    calc_point_agb(pts_samp)

    # Join points to hedge polygons so points carry IndexerKey
    arcpy.analysis.SpatialJoin(
        target_features=pts_samp,
        join_features=hedge_poly,
        out_feature_class=pt_poly,
        join_operation="JOIN_ONE_TO_MANY",
        join_type="KEEP_COMMON",
        match_option="WITHIN"
    )

    # Stats by IndexerKey
    arcpy.analysis.Statistics(
        in_table=pt_poly,
        out_table=sum_tbl,
        statistics_fields=[["AGB_KG_PT", "SUM"], ["AGB_KG_PT", "COUNT"]],
        case_field=[KEY_FIELD]
    )

    lk_sum, lk_cnt = {}, {}
    with arcpy.da.SearchCursor(sum_tbl, [KEY_FIELD, "SUM_AGB_KG_PT", "COUNT_AGB_KG_PT"]) as cur:
        for k, s, c in cur:
            if k is None:
                continue
            k = int(k)
            lk_sum[k] = 0.0 if s is None else float(s)
            lk_cnt[k] = 0.0 if c is None else float(c)

    for k in sum_agb_by_key.keys():
        s = lk_sum.get(k, 0.0)
        c = lk_cnt.get(k, 0.0)
        if c <= 0:
            continue

        if SCALE_TO_AREA:
            expected = expN_by_key.get(k, 0.0)
            v = s * (expected / c) if expected > 0 else 0.0
        else:
            v = s

        sum_agb_by_key[k] += v

    valid_iters += 1

if valid_iters == 0:
    raise RuntimeError("No valid iterations completed (no points or no valid CHM samples).")

mean_agb_by_key = {k: (v / valid_iters) for k, v in sum_agb_by_key.items()}

# ----------------------------
# 4) Write back to fc_all by IndexerKey (robust)
# ----------------------------
total_written = 0.0
n_updated = 0
n_missing_keys = 0

with arcpy.da.UpdateCursor(fc_all, [KEY_FIELD, LC_FIELD, AGB_FIELD]) as cur:
    for k, lc, agb in cur:
        if k is None:
            continue

        # LC==23 check with correct type
        is_hedge = (str(lc) == str(HEDGE_LC)) if lc_is_text else (lc == HEDGE_LC)
        if not is_hedge:
            continue

        if ONLY_IF_NULL and agb is not None and not (TREAT_ZERO_AS_MISSING and float(agb) == 0.0):
            continue

        k = int(k)
        if k not in mean_agb_by_key:
            # no sampled points / no valid CHM samples for this key
            n_missing_keys += 1
            continue

        v = float(mean_agb_by_key[k])
        cur.updateRow([k, lc, v])
        total_written += v
        n_updated += 1

print("\n✅ DONE")
print(f"Updated LC=23 features: {n_updated:,}")
print(f"Total hedge AGB written: {total_written:,.1f} kg (= {total_written/1000.0:,.2f} t)")
print(f"LC=23 rows still missing (no samples/NoData): {n_missing_keys:,}")

# ----------------------------
# 5) Cleanup
# ----------------------------
if not KEEP_INTERMEDIATES:
    for p in [hedge_poly, pts_rand, pts_samp, pt_poly, sum_tbl]:
        safe_delete(p)

print("Finished.")


Where clause: LC = 23
Hedge polygons selected: 4,559
Hedge area (m²): 164,501.2
Requested points per iter: 200,000  (spacing=1.0 m, safety=0.25)


Hedge AGB iters: 100%|██████████| 20/20 [32:19<00:00, 96.97s/it] ﻿



✅ DONE
Updated LC=23 features: 4,559
Total hedge AGB written: 1,262,614.1 kg (= 1,262.61 t)
LC=23 rows still missing (no samples/NoData): 0
Finished.


Now AGB for trees

In [10]:
import os
import math
import arcpy

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

# =============================================================================
# TREE AGB FROM CROWN POLYGONS (LC MAPPING)
# LC=1 broadleaf, LC=2 conifer, LC=23 ignore, all other LCs treated as broadleaf
# - CD from polygon area
# - DBH from Eq2
# - AGB from Eq3 (broadleaf), Eq4 (conifer), Eq5 (rho-based, optional)
# - Writes AGB_TREE_KG as the "chosen" equation result (Eq3 or Eq4 by LC mapping)
# =============================================================================

arcpy.env.overwriteOutput = True

# ----------------------------
# INPUTS
# ----------------------------
gdb = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb"
fc  = os.path.join(gdb, "Crowns_plus_hedges_nooverlap")

# ----------------------------
# FIELDS
# ----------------------------
LC_FIELD = "LC"
H_FIELD  = "H_M"        # height in metres (must exist)

# Outputs (created if missing)
CD_M_FIELD     = "CD_M"
DBH_CM_FIELD   = "DBH_CM"
RHO_FIELD      = "RHO_GCM3"
AGB_EQ3_FIELD  = "AGB_EQ3_KG"
AGB_EQ4_FIELD  = "AGB_EQ4_KG"
AGB_EQ5_FIELD  = "AGB_EQ5_KG"
AGB_TREE_FIELD = "AGB_TREE_KG"   # <- final chosen AGB for trees (Eq3 or Eq4)

# ----------------------------
# LC RULES (YOUR REQUEST)
# ----------------------------
LC_BROAD = 1
LC_CONIF = 2
LC_HEDGE = 23

# Eq5 wood density (only affects Eq5)
RHO_BROAD = 0.610
RHO_CONIF = 0.510

# ----------------------------
# EQUATIONS (as previously used)
# ----------------------------
# Eq2: DBH(cm) = (k * (H(m)*CD(cm))^p) * exp(sigma^2/2)
EQ2_K = 0.557
EQ2_P = 0.809
EQ2_SIGMA = 0.056

# Eq3 (broadleaf): AGB = 0.08 + (25000*DBH^2.5)/(DBH^2.5 + 246872)
# Eq4 (conifer):   AGB = 0.022*DBH^2.73 + 0.19*H^2.06
# Eq5 (generic):   AGB = (0.0673*(rho*DBH*H)^0.976) * exp(0.357^2/2)
EQ5_A = 0.0673
EQ5_P = 0.976
EQ5_SIGMA = 0.357

# ----------------------------
# GUARDS
# ----------------------------
MIN_AREA_M2 = 0.01
MIN_H_M     = 0.5

# ----------------------------
# Helpers
# ----------------------------
def ensure_field(in_fc, name, ftype, length=None):
    existing = {f.name.upper() for f in arcpy.ListFields(in_fc)}
    if name.upper() in existing:
        return
    if ftype.upper() == "TEXT" and length:
        arcpy.management.AddField(in_fc, name, ftype, field_length=length)
    else:
        arcpy.management.AddField(in_fc, name, ftype)

def crown_diameter_m(area_m2):
    return 2.0 * math.sqrt(area_m2 / math.pi)

def dbh_cm_eq2(h_m, cd_cm):
    base = EQ2_K * ((h_m * cd_cm) ** EQ2_P)
    return base * math.exp((EQ2_SIGMA ** 2) / 2.0)

def agb_eq3(dbh_cm):
    x = dbh_cm ** 2.5
    return 0.08 + (25000.0 * x) / (x + 246872.0)

def agb_eq4(dbh_cm, h_m):
    return (0.022 * (dbh_cm ** 2.73)) + (0.19 * (h_m ** 2.06))

def agb_eq5(dbh_cm, h_m, rho):
    return (EQ5_A * ((rho * dbh_cm * h_m) ** EQ5_P)) * math.exp((EQ5_SIGMA ** 2) / 2.0)

def parse_lc(lc_val, lc_is_text):
    """Return int LC where possible; None if missing/unparseable."""
    if lc_val is None:
        return None
    if lc_is_text:
        s = str(lc_val).strip()
        if s == "":
            return None
        try:
            return int(float(s))
        except Exception:
            return None
    else:
        try:
            return int(lc_val)
        except Exception:
            return None

# ----------------------------
# Checks + add fields
# ----------------------------
if not arcpy.Exists(fc):
    raise RuntimeError(f"Missing feature class: {fc}")

fields = {f.name.upper(): f for f in arcpy.ListFields(fc)}
for req in [LC_FIELD, H_FIELD]:
    if req.upper() not in fields:
        raise RuntimeError(f"Missing required field: {req}")

lc_type = fields[LC_FIELD.upper()].type.lower()
lc_is_text = lc_type in ("string", "text")

ensure_field(fc, CD_M_FIELD, "DOUBLE")
ensure_field(fc, DBH_CM_FIELD, "DOUBLE")
ensure_field(fc, RHO_FIELD, "DOUBLE")
ensure_field(fc, AGB_EQ3_FIELD, "DOUBLE")
ensure_field(fc, AGB_EQ4_FIELD, "DOUBLE")
ensure_field(fc, AGB_EQ5_FIELD, "DOUBLE")
ensure_field(fc, AGB_TREE_FIELD, "DOUBLE")

# ----------------------------
# Loop + compute
# ----------------------------
count_total = int(arcpy.management.GetCount(fc)[0])

total_eq3 = 0.0
total_eq4 = 0.0
total_eq5 = 0.0
total_tree = 0.0

n_tree = 0
n_hedge = 0
n_invalid = 0

# IMPORTANT: keep cursor object separate from tqdm wrapper
cur = arcpy.da.UpdateCursor(
    fc,
    [LC_FIELD, H_FIELD, "SHAPE@AREA",
     CD_M_FIELD, DBH_CM_FIELD, RHO_FIELD,
     AGB_EQ3_FIELD, AGB_EQ4_FIELD, AGB_EQ5_FIELD, AGB_TREE_FIELD]
)

it = cur
if tqdm:
    it = tqdm(cur, total=count_total, desc="Tree AGB (Eq2+Eq3/Eq4/Eq5)")

for row in it:
    lc_raw, h_m, area_m2 = row[0], row[1], row[2]

    lc_int = parse_lc(lc_raw, lc_is_text)

    # Ignore hedges entirely
    if lc_int == LC_HEDGE:
        n_hedge += 1
        continue

    # Need valid geometry area + height
    if area_m2 is None or area_m2 <= MIN_AREA_M2 or h_m is None or h_m < MIN_H_M:
        # blank outputs for non-hedge invalid rows
        row[3] = None  # CD_M
        row[4] = None  # DBH_CM
        row[5] = None  # RHO
        row[6] = None  # EQ3
        row[7] = None  # EQ4
        row[8] = None  # EQ5
        row[9] = None  # TREE
        cur.updateRow(row)
        n_invalid += 1
        continue

    # LC mapping: treat anything except LC=2 as broadleaf
    is_conifer = (lc_int == LC_CONIF)

    # Crown diameter
    cd_m = crown_diameter_m(float(area_m2))
    cd_cm = cd_m * 100.0

    # DBH from Eq2
    dbh_cm = dbh_cm_eq2(float(h_m), cd_cm)

    # Eq5 rho
    rho = RHO_CONIF if is_conifer else RHO_BROAD

    # AGB equations
    eq3 = agb_eq3(dbh_cm)
    eq4 = agb_eq4(dbh_cm, float(h_m))
    eq5 = agb_eq5(dbh_cm, float(h_m), rho)

    # Final chosen tree AGB (your rule)
    tree_agb = eq4 if is_conifer else eq3

    # Write back
    row[3] = cd_m
    row[4] = dbh_cm
    row[5] = rho
    row[6] = eq3
    row[7] = eq4
    row[8] = eq5
    row[9] = tree_agb
    cur.updateRow(row)

    # Totals
    total_eq3 += float(eq3)
    total_eq4 += float(eq4)
    total_eq5 += float(eq5)
    total_tree += float(tree_agb)
    n_tree += 1

# ----------------------------
# Print totals
# ----------------------------
print("\n================ TREE AGB TOTALS ================")
print(f"Processed (non-hedge) features: {n_tree:,}")
print(f"Ignored hedges (LC=23):         {n_hedge:,}")
print(f"Invalid/non-tree rows:          {n_invalid:,}\n")

print(f"Trees total Eq3 (kg): {total_eq3:,.0f}  ({total_eq3/1000.0:,.1f} t)")
print(f"Trees total Eq4 (kg): {total_eq4:,.0f}  ({total_eq4/1000.0:,.1f} t)")
print(f"Trees total Eq5 (kg): {total_eq5:,.0f}  ({total_eq5/1000.0:,.1f} t)")

print("\n---- FINAL (YOUR LC RULE) ----")
print("Tree AGB used = Eq4 if LC=2 else Eq3 (LC=23 ignored)")
print(f"Trees total AGB_TREE_KG (kg): {total_tree:,.0f}  ({total_tree/1000.0:,.1f} t)")
print("===============================================")


Tree AGB (Eq2+Eq3/Eq4/Eq5): 100%|██████████| 31853/31853 [00:03<00:00, 7972.90it/s] ﻿



================ TREE AGB TOTALS ================
Processed (non-hedge) features: 27,267
Ignored hedges (LC=23):         4,559
Invalid/non-tree rows:          27

Trees total Eq3 (kg): 630,478,846  (630,478.8 t)
Trees total Eq4 (kg): 88,929,472,795  (88,929,472.8 t)
Trees total Eq5 (kg): 8,114,795  (8,114.8 t)

---- FINAL (YOUR LC RULE) ----
Tree AGB used = Eq4 if LC=2 else Eq3 (LC=23 ignored)
Trees total AGB_TREE_KG (kg): 2,306,445,640  (2,306,445.6 t)


**Print results

In [13]:
import os
import arcpy

# ----------------------------
# INPUTS
# ----------------------------
gdb = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb"
fc  = os.path.join(gdb, "Crowns_plus_hedges_nooverlap")

LC_FIELD  = "LC"
AGB_FIELD = "AGB_KG"
HEDGE_LC  = 23

# ----------------------------
# HELPERS
# ----------------------------
def is_lc23(v):
    if v is None:
        return False
    try:
        return int(v) == HEDGE_LC
    except Exception:
        try:
            return int(float(str(v).strip())) == HEDGE_LC
        except Exception:
            return str(v).strip() == str(HEDGE_LC)

def pct(vals, p):
    if not vals:
        return None
    vals = sorted(vals)
    k = (len(vals) - 1) * (p / 100.0)
    f = int(k)
    c = min(f + 1, len(vals) - 1)
    if c == f:
        return vals[f]
    return vals[f] + (vals[c] - vals[f]) * (k - f)

# ----------------------------
# ACCUMULATORS
# ----------------------------
groups = {
    "HEDGES (LC=23)": {"n": 0, "agb_kg": 0.0, "area_m2": 0.0, "rat_kg_ha": []},
    "TREES (LC!=23)": {"n": 0, "agb_kg": 0.0, "area_m2": 0.0, "rat_kg_ha": []},
}
n_bad_geom = 0
n_zero_area = 0

# ----------------------------
# LOOP
# ----------------------------
fields = [LC_FIELD, AGB_FIELD, "SHAPE@AREA", "SHAPE@"]

with arcpy.da.SearchCursor(fc, fields) as cur:
    for lc, agb, area_m2, geom in cur:
        if geom is None:
            n_bad_geom += 1
            continue
        if area_m2 is None or float(area_m2) <= 0:
            n_zero_area += 1
            continue

        gname = "HEDGES (LC=23)" if is_lc23(lc) else "TREES (LC!=23)"
        g = groups[gname]

        agb_kg = 0.0 if agb is None else float(agb)
        area_m2 = float(area_m2)

        # per-feature ratio
        kg_ha = (agb_kg / area_m2) * 10000.0

        g["n"] += 1
        g["agb_kg"] += agb_kg
        g["area_m2"] += area_m2
        g["rat_kg_ha"].append(kg_ha)

# ----------------------------
# REPORT
# ----------------------------
print("\n================ AGB / AREA (PER FEATURE) =================")
print(f"Feature class: {fc}")

if n_bad_geom or n_zero_area:
    print(f"Skipped: null geometry={n_bad_geom:,} | zero/invalid area={n_zero_area:,}")

for name, g in groups.items():
    n = g["n"]
    if n == 0:
        print(f"\n{name}: n=0")
        continue

    total_agb = g["agb_kg"]
    total_area_ha = g["area_m2"] / 10000.0

    # area-weighted (total agb / total area)
    weighted_kg_ha = (total_agb / g["area_m2"]) * 10000.0
    weighted_t_ha  = weighted_kg_ha / 1000.0

    # unweighted mean of per-feature ratios
    mean_kg_ha = sum(g["rat_kg_ha"]) / n
    mean_t_ha  = mean_kg_ha / 1000.0

    p50 = pct(g["rat_kg_ha"], 50)
    p95 = pct(g["rat_kg_ha"], 95)

    print(f"\n{name}")
    print(f"  n={n:,} | total_area={total_area_ha:,.2f} ha | total_agb={total_agb:,.1f} kg ({total_agb/1000.0:,.2f} t)")
    print(f"  total_agb/total_area: {weighted_kg_ha:,.2f} kg/ha ({weighted_t_ha:,.3f} t/ha)")
    print(f"  mean(agb/area per feature): {mean_kg_ha:,.2f} kg/ha ({mean_t_ha:,.3f} t/ha)")
    print(f"  per-feature kg/ha: p50={p50:,.2f} | p95={p95:,.2f}")

print("===========================================================\n")



================ AGB / AREA (PER FEATURE) =================
Feature class: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb\Crowns_plus_hedges_nooverlap
Skipped: null geometry=26 | zero/invalid area=0

HEDGES (LC=23)
  n=4,559 | total_area=16.45 ha | total_agb=1,262,614.1 kg (1,262.61 t)
  total_agb/total_area: 76,754.09 kg/ha (76.754 t/ha)
  mean(agb/area per feature): 50,310.63 kg/ha (50.311 t/ha)
  per-feature kg/ha: p50=1,335.14 | p95=217,786.53

TREES (LC!=23)
  n=27,268 | total_area=169.88 ha | total_agb=6,879,694.6 kg (6,879.69 t)
  total_agb/total_area: 40,497.01 kg/ha (40.497 t/ha)
  mean(agb/area per feature): 68,577.63 kg/ha (68.578 t/ha)
  per-feature kg/ha: p50=53,697.99 | p95=174,928.10



# 